# Lean/Mathlib cache for Colab + Google Drive

This notebook keeps a compressed Lean/Mathlib environment in Google Drive, restores it to Colab local disk at runtime, and runs Lean from local disk for speed.

Storage model:

- Google Drive: persistent archive `lean-mathlib-cache.tar.zst`
- Colab local disk: active runtime directory `/content/lean-env`

Run the cells from top to bottom. The first run may take a while because it installs Lean and fetches Mathlib cache. Later runs should mostly restore the archive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, pathlib

DRIVE_CACHE_DIR = pathlib.Path('/content/drive/MyDrive/lean-cache')
ARCHIVE = DRIVE_CACHE_DIR / 'lean-mathlib-cache.tar.zst'
LEAN_ENV = pathlib.Path('/content/lean-env')
PROJECT = LEAN_ENV / 'bruhat_mathlib'
ELAN_HOME = LEAN_ENV / 'elan'

DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
print('Drive cache dir:', DRIVE_CACHE_DIR)
print('Archive:', ARCHIVE)
print('Runtime dir:', LEAN_ENV)
print('Archive exists:', ARCHIVE.exists())

In [ ]:
%%bash
set -euxo pipefail
apt-get update -y
apt-get install -y curl git zstd ca-certificates

## Restore from Drive if the archive already exists

This copies the archive to Colab local disk first, then extracts locally. That is usually faster than extracting directly from the Drive mount.

In [ ]:
import os, shutil, subprocess

if ARCHIVE.exists():
    local_archive = pathlib.Path('/content/lean-mathlib-cache.tar.zst')
    if LEAN_ENV.exists():
        shutil.rmtree(LEAN_ENV)
    print('Copying archive from Drive to local disk...')
    shutil.copy2(ARCHIVE, local_archive)
    print('Extracting archive...')
    subprocess.run(['tar', '--zstd', '-xf', str(local_archive), '-C', '/content'], check=True)
    print('Restored:', LEAN_ENV.exists())
else:
    print('No archive found. The next cell will build the Lean/Mathlib environment once.')

## First-time build if no archive exists

This installs Lean through elan, creates a Mathlib project, downloads precompiled Mathlib cache with `lake exe cache get`, tests `import Mathlib`, then saves the whole environment back to Google Drive.

In [ ]:
%%bash
set -euxo pipefail

ARCHIVE='/content/drive/MyDrive/lean-cache/lean-mathlib-cache.tar.zst'
LEAN_ENV='/content/lean-env'
ELAN_HOME="$LEAN_ENV/elan"
PROJECT="$LEAN_ENV/bruhat_mathlib"

if [ ! -f "$ARCHIVE" ]; then
  rm -rf "$LEAN_ENV"
  mkdir -p "$LEAN_ENV"
  export ELAN_HOME="$ELAN_HOME"
  export PATH="$ELAN_HOME/bin:$PATH"

  curl -sSfL https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh | sh -s -- -y --no-modify-path

  cd "$LEAN_ENV"
  lake +leanprover-community/mathlib4:lean-toolchain new bruhat_mathlib math
  cd "$PROJECT"
  lake update
  lake exe cache get

  cat > BruhatSmokeTest.lean <<'EOF'
import Mathlib

example (n : Nat) : n = n := rfl
EOF
  lake env lean BruhatSmokeTest.lean

  cd /content
  tar --zstd -cf /content/lean-mathlib-cache.tar.zst lean-env
  cp /content/lean-mathlib-cache.tar.zst "$ARCHIVE"
else
  echo "Archive already exists; skipping first-time build."
fi

## Activate Lean for this Colab session

Run this after either restoring or building. It prints the Lean version and verifies that Mathlib imports.

In [ ]:
import os, subprocess, pathlib

os.environ['ELAN_HOME'] = str(ELAN_HOME)
os.environ['PATH'] = f"{ELAN_HOME}/bin:" + os.environ['PATH']

assert PROJECT.exists(), f'Missing project: {PROJECT}'
subprocess.run(['lean', '--version'], check=True)

smoke = PROJECT / 'BruhatSmokeTest.lean'
smoke.write_text('import Mathlib\n\nexample (n : Nat) : n = n := rfl\n')
subprocess.run(['lake', 'env', 'lean', str(smoke.name)], cwd=PROJECT, check=True)
print('LEAN_MATHLIB_OK')

## Verify your own Lean file

Upload a `.lean` file to Colab, or copy it into the project directory, then set `LEAN_FILE` below. If the file uses Mathlib, keep it inside the project directory and run it with `lake env lean`.

In [ ]:
LEAN_FILE = PROJECT / 'BruhatSmokeTest.lean'
subprocess.run(['lake', 'env', 'lean', str(LEAN_FILE.name)], cwd=PROJECT, check=True)
print('VERIFY_OK:', LEAN_FILE)

## Repack after changes

Only run this if you intentionally changed the Lean environment or project and want to update the Drive archive.

In [ ]:
import subprocess, shutil

local_archive = pathlib.Path('/content/lean-mathlib-cache.tar.zst')
subprocess.run(['tar', '--zstd', '-cf', str(local_archive), '-C', '/content', 'lean-env'], check=True)
shutil.copy2(local_archive, ARCHIVE)
print('Updated archive:', ARCHIVE)